# Debugging
## Week 4 - Discussion Section

### 1. Load the data and aggregate

In [20]:
import io
import pandas as pd

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV), na_values=-999.0)

In [21]:
stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,NaN,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


Before writing any more code, agree with your team on the answers to these three questions and write them in a markdown cell.

How many monitoring sites are in this dataset?
- Probably 3: SB-01, SB-02, SB-03
What is a plausible range for stream water temperature in degrees Celsius?
- 10-25 degC
Roughly what mean water temperature do you expect across all measurements?
- 16 degC

### 2. A worked example

In [ ]:
# Convert stream discharge from cubic feet per second to cubic metres per second
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [4]:
# Display column types
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [ ]:
# Reduce to smallest failure model of issue
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

TypeError: can't multiply sequence by non-int of type 'float'

In [6]:
# Examine unique values
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [7]:
# Remove "," separator in discharge_cmf column, convert to float, perform conversion
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False)
                    .astype(float)) * 0.0283168

discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [8]:
# Add guard against this issue
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [9]:
# Reproducible example to ask for help
import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'

| What I expected | What happened | My hypothesis | What I changed | Result |
| --------------- | ------------- | ------------- | -------------- | ------ |
| Discharge in m³/s, around 35 | TypeError: can't multiply sequence by non-int | discharge_cfs is text because of thousands separators | .str.replace(',', '') then .astype(float) | Works, 35.11 m³/s, guard added |
| site_id w/o spaces | AttributeError: 'Series' object has no attribute 'strip' | str.strip() is the Series method for this, not .strip() | .str.strip() | Works, strips spaces from site_id Series |
| Convert nitrate_mg_l column to float | ValueError: could not convert string to float: '<0.05' | "<" chr in column prevents type coercion | Added .str.strip("<").astype(float) | Works, column is now float. |

a.

Read the error, or state the mismatch. Note the error type, the message, and the last line of the traceback that points at your code.

Look at the data. Check .shape, .dtypes, .columns, and .head(). Most bugs are a mismatch between the data you have and the data you assumed.

Isolate. Reduce the failing code to the smallest piece that still breaks.

Form a hypothesis and test it. State what you think is wrong in one sentence, then write code that confirms or rejects it.

Read the documentation for any function or method involved, because the answer is very often written there already.

Fix, then verify. Re-run from the top and confirm the fix did not break something downstream.

Consider adding a guard

In [10]:
stream['site_id'].strip()

AttributeError: 'Series' object has no attribute 'strip'

In [ ]:
type(stream['site_id'].iloc[0]).strip() # iloc actually returns a str!

str

In [13]:
# Need to use str.strip() method
stream['site_id'].str.strip()

0     SB-01
1     SB-01
2     SB-01
3     SB-01
4     sb-01
5     SB-01
6     SB-02
7     SB-02
8     SB-02
9     SB-02
10    SB-02
11    sb-02
12    SB-03
13    SB-03
14    SB-03
15    SB-03
16    SB-03
17    SB-03
Name: site_id, dtype: object

b.

Read the error, or state the mismatch. Note the error type, the message, and the last line of the traceback that points at your code.

Look at the data. Check .shape, .dtypes, .columns, and .head(). Most bugs are a mismatch between the data you have and the data you assumed.

Isolate. Reduce the failing code to the smallest piece that still breaks.

Form a hypothesis and test it. State what you think is wrong in one sentence, then write code that confirms or rejects it.

Read the documentation for any function or method involved, because the answer is very often written there already.

Fix, then verify. Re-run from the top and confirm the fix did not break something downstream.

Consider adding a guard

In [14]:
# Error code
stream['nitrate_mg_l'].astype(float)

ValueError: could not convert string to float: '<0.05'

In [ ]:
# Look at column type
print(stream['nitrate_mg_l'].dtype)
# Look at unique values
stream['nitrate_mg_l'].unique()
# "<" chr in column prevents type coercion


object


array(['0.82', '0.79', '1.21', '0.94', '0.66', '0.51', '1.88', '1.92',
       '2.41', '2.09', '1.44', '1.02', '<0.05', '0.33', '0.28'],
      dtype=object)

In [18]:
stream['nitrate_mg_l'].str.strip("<").astype(float)

0     0.82
1     0.79
2     1.21
3     0.94
4     0.66
5     0.51
6     1.88
7     1.92
8     2.41
9     2.09
10    1.44
11    1.02
12    0.05
13    0.05
14    0.33
15    0.28
16    0.05
17    0.05
Name: nitrate_mg_l, dtype: float64

### 4. Bugs that do not raise errors

a. Run the following cell and compare the result against your expectation from 1c-iii.

In [ ]:
# Adding na_values = -999.0 to read_csv() step
stream['water_temp_c'].mean()

15.226666666666668

b. Run the following cell and compare the result against your expectation from 1c-i.

In [24]:
stream['site_id'].nunique()

7

In [ ]:
# Some rows are lowercase
stream['site_id'].unique()

array(['SB-01', 'sb-01', 'SB-01 ', 'SB-02', 'sb-02', 'SB-03', 'SB-03 '],
      dtype=object)

In [29]:
# Fix
stream['site_id'] = stream['site_id'].str.strip().str.upper()

In [ ]:
# Confirm
stream['site_id'].nunique()

3